# Notebook 5 — GraphRAG in Practice: Compare Retrieval Strategies

This is the **final step** of the GraphRAG pipeline. It demonstrates three different RAG strategies over the knowledge base built in Notebooks 1–4 and lets you compare their answers side-by-side.

## The three retrieval modes

| Mode | Retrieval source | What it's good at |
|------|-----------------|-------------------|
| 🔍 **Vector only** | Qdrant semantic similarity | Finding passages that are topically similar to the question — fast, broad recall |
| 🕸️ **Graph only** | Neo4j multi-hop traversal | Discovering explicit connections and chains between entities (e.g. A influenced B who worked with C) |
| ⚡ **Hybrid** | Vector + Graph combined | The most complete answers — semantic context plus structured relational facts |

## Query pipeline (for each question)

```
Question
  │
  ├─► Entity extraction (GPT-4o) ──► entity names + types
  │
  ├─► Qdrant search ──► top-5 most similar text chunks
  │
  ├─► Neo4j search ──► 1-hop + 2-hop relationship chains
  │       (using extracted entities + article titles from vector results)
  │
  ├─► Answer 1: Vector-only context → GPT-4o
  ├─► Answer 2: Graph-only context  → GPT-4o
  └─► Answer 3: Hybrid context      → GPT-4o
```

## Prerequisites
- Notebooks 1–3 completed (Neo4j populated)
- Notebook 4 completed (Qdrant populated)
- Neo4j running at `bolt://localhost:7687`
- Qdrant running at `http://localhost:6333`

## 1 · Imports

In [21]:
import json
import os
import re
from pathlib import Path
from dotenv import load_dotenv
from openai import AzureOpenAI
from qdrant_client import QdrantClient
from neo4j import GraphDatabase

## 2 · Configuration

In [12]:
SCRIPT_DIR = Path(".").resolve()
load_dotenv(dotenv_path=SCRIPT_DIR.parent / ".env")

# Azure OpenAI
endpoint            = os.getenv("endpoint")
subscription_key    = os.getenv("subscription_key")
deployment          = os.getenv("deployment", "gpt-4o")
api_version         = os.getenv("api_version", "2024-12-01-preview")
embedding_deployment = os.getenv("embedding_deployment", "text-embedding-3-small")
embedding_api_version = os.getenv("embedding_api_version", "2024-02-01")

# Neo4j
NEO4J_URI      = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER     = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

# Qdrant
QDRANT_URL       = "http://localhost:6333"
COLLECTION_NAME  = "wikipedia_scientists"
QDRANT_TOP_K     = 5

# Entity and relationship types found in the knowledge graph
ENTITY_TYPES = [
    "Animal", "Award", "Concept", "Country", "Discovery", "Disease",
    "Element", "Event", "Field", "Film", "Group", "Invention", "Law",
    "Object", "Organization", "Person", "Place", "Position", "Project",
    "Publication", "Substance", "Technology", "Theorem", "Theory", "Unit",
]

RELATIONSHIP_TYPES = [
    "ADVISED", "AFFILIATED_WITH", "ANALYZED", "ATTRIBUTED_TO", "AWARDED",
    "BORN_IN", "CHILD_OF", "COLLABORATED_WITH", "COLLEAGUE_OF",
    "COMMENTED_ON", "CONFIRMED", "CONTEMPORARY_OF", "CONTRIBUTED_TO",
    "CORRESPONDED_WITH", "CRITICIZED", "DEFENDED", "DEVELOPED", "DEVISED",
    "DIED_IN", "DIRECTED", "DISCOVERED", "DISPROVED", "EDITED",
    "EDUCATED_AT", "ELECTED_TO", "EMPLOYED_AT", "EULOGIZED", "FOUNDED",
    "FRIEND_OF", "FUNDED", "GRADUATED_FROM", "GREW_UP_IN", "HIRED_BY",
    "HONORED", "HONORED_WITH", "INFLUENCED", "INFLUENCED_BY", "INSPIRED_BY",
    "INVENTED", "KNIGHTED_BY", "KNOWN_FOR", "LED", "LICENSED_FROM",
    "LICENSED_TO", "MARRIED_TO", "MEMBER_OF", "MENTORED", "MENTORED_BY",
    "MURDERED", "NAMED_AFTER", "PARENT_OF", "PREDICTED", "PUBLISHED",
    "RECEIVED", "RECOGNIZED_BY", "REDISCOVERED", "RELATED_TO", "RIVAL_OF",
    "SIBLING_OF", "STUDENT_OF", "STUDIED_AT", "STUDIED_UNDER",
    "STUDIED_WITH", "SUPPORTED", "SUPPORTED_BY", "TAUGHT_AT", "TRANSLATED",
    "VERIFIED", "WON", "WORKED_AT", "WORKED_IN", "WORKED_ON", "WORKED_WITH",
    "WROTE_ABOUT", "WROTE_TO",
]

print("Configuration loaded.")


Configuration loaded.


## 3 · Initialise Clients

In [13]:
openai_client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
    azure_deployment=deployment,
)

embedding_client = AzureOpenAI(
    api_version=embedding_api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
    azure_deployment=embedding_deployment,
)

qdrant = QdrantClient(url=QDRANT_URL)
neo4j_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

print("Clients initialised.")

Clients initialised.


## 4 · Entity Extraction

In [14]:
def extract_entities_from_question(question: str) -> list[dict]:
    """Use the LLM to identify named entities in the user's question."""
    entity_list = ", ".join(ENTITY_TYPES)
    prompt = f"""You are a named entity extractor.

Extract all named entities from the question below.
Only return entities whose type is one of: {entity_list}

Return ONLY a JSON array of objects with "name" and "type" fields.
No markdown, no explanation.

Question: {question}

JSON array:"""

    response = openai_client.chat.completions.create(
        model=deployment,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    raw = response.choices[0].message.content.strip()
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)
    try:
        entities = json.loads(raw)
        return entities if isinstance(entities, list) else []
    except json.JSONDecodeError:
        print(f"  [warn] Could not parse entity extraction response: {raw}")
        return []

## 5 · Vector Search (Qdrant)

In [15]:
def search_qdrant(question: str, top_k: int = QDRANT_TOP_K) -> list[dict]:
    """Embed the question and retrieve the top-k most similar chunks from Qdrant."""
    response = embedding_client.embeddings.create(
        input=question,
        model=embedding_deployment,
    )
    query_vector = response.data[0].embedding

    results = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
        with_payload=True,
    )

    return [
        {
            "title":       hit.payload.get("title", ""),
            "chunk_index": hit.payload.get("chunk_index", 0),
            "text":        hit.payload.get("text", ""),
            "score":       hit.score,
        }
        for hit in results.points
    ]

## 6 · Graph Search (Neo4j)

In [16]:
def search_neo4j(entities: list[dict]) -> list[dict]:
    """
    For each entity run two queries:
    - 1-hop: direct facts (relationships to immediate neighbours)
    - 2-hop: chains through intermediate nodes
    """
    if not entities:
        return []

    results = []
    with neo4j_driver.session(database=NEO4J_DATABASE) as session:
        for entity in entities:
            name = entity.get("name", "")
            if not name:
                continue

            # 1-hop direct facts
            direct = session.run(
                """
                MATCH (e:Entity)-[r]-(n:Entity)
                WHERE toLower(e.name) CONTAINS toLower($name)
                RETURN e.name AS source, type(r) AS relationship,
                       n.name AS target, n.type AS target_type
                LIMIT 30
                """,
                name=name,
            ).data()

            # 2-hop chains
            paths = session.run(
                """
                MATCH (e:Entity)-[r1]-(mid:Entity)-[r2]-(n:Entity)
                WHERE toLower(e.name) CONTAINS toLower($name)
                  AND mid.name <> e.name
                  AND n.name <> e.name
                RETURN e.name AS source,
                       type(r1) AS rel1,
                       mid.name AS mid_node, mid.type AS mid_type,
                       type(r2) AS rel2,
                       n.name AS target, n.type AS target_type
                LIMIT 40
                """,
                name=name,
            ).data()

            if direct or paths:
                results.append({
                    "query_entity": name,
                    "direct":       direct,
                    "paths":        paths,
                })

    return results

## 7 · Context Builders & LLM Helper

In [17]:
def build_vector_context(qdrant_results: list[dict]) -> str:
    parts = ["## Relevant Wikipedia Passages\n"]
    for i, r in enumerate(qdrant_results, 1):
        parts.append(
            f"[{i}] {r['title']} (chunk {r['chunk_index']}, score {r['score']:.3f})\n{r['text']}\n"
        )
    return "\n".join(parts)


def build_graph_context(neo4j_results: list[dict]) -> str:
    parts = ["## Knowledge Graph Facts\n"]
    parts.append(
        "Direct connections are single-hop facts. "
        "Multi-hop chains follow relationship paths through intermediate nodes — "
        "indirect connections that vector search cannot discover.\n"
    )
    for entry in neo4j_results:
        parts.append(f"### {entry['query_entity']}")
        if entry.get("direct"):
            parts.append("Direct connections:")
            for t in entry["direct"]:
                parts.append(
                    f"  {t['source']} --[{t['relationship']}]--> {t['target']} ({t['target_type']})"
                )
        if entry.get("paths"):
            parts.append("Multi-hop chains (2 hops):")
            for p in entry["paths"]:
                parts.append(
                    f"  {p['source']} --[{p['rel1']}]--> "
                    f"{p['mid_node']} ({p['mid_type']}) "
                    f"--[{p['rel2']}]--> {p['target']} ({p['target_type']})"
                )
        parts.append("")
    return "\n".join(parts)


def _call_llm(question: str, context: str, system_prompt: str) -> str:
    response = openai_client.chat.completions.create(
        model=deployment,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content.strip()


def _collect_graph_entities(entities: list[dict], qdrant_results: list[dict]) -> list[dict]:
    """Merge question entities with Qdrant article titles for graph lookup."""
    seen = {e["name"].lower() for e in entities}
    graph_entities = list(entities)
    for chunk in qdrant_results:
        title = chunk["title"]
        if title.lower() not in seen:
            graph_entities.append({"name": title, "type": "Person"})
            seen.add(title.lower())
    return graph_entities

## 8 · Answer Modes

In [18]:
VECTOR_SYSTEM_PROMPT = "You are a knowledgeable assistant. Answer the question concisely and factually."
GRAPH_SYSTEM_PROMPT  = "You are a knowledgeable assistant. Answer the question concisely and factually."
HYBRID_SYSTEM_PROMPT = "You are a knowledgeable assistant. Answer the question concisely and factually."


def answer_vector_only(question: str, entities: list[dict], qdrant_results: list[dict]) -> str:
    context = build_vector_context(qdrant_results)
    return _call_llm(question, context, VECTOR_SYSTEM_PROMPT)


def answer_graph_only(question: str, neo4j_results: list[dict]) -> str:
    context = build_graph_context(neo4j_results)
    return _call_llm(question, context, GRAPH_SYSTEM_PROMPT)


def answer_hybrid(question: str, qdrant_results: list[dict], neo4j_results: list[dict]) -> str:
    context = build_vector_context(qdrant_results) + "\n" + build_graph_context(neo4j_results)
    return _call_llm(question, context, HYBRID_SYSTEM_PROMPT)

## 9 · Compare All Modes

In [19]:
def compare_answers(question: str) -> None:
    sep  = "=" * 80
    thin = "-" * 80

    print(f"\n{sep}")
    print(f"QUESTION: {question}")
    print(sep)

    # Shared retrieval — done once, reused by all three modes
    print("  Extracting entities...")
    entities = extract_entities_from_question(question)
    print(f"    Found: {[e['name'] for e in entities]}")

    print("  Searching Qdrant...")
    qdrant_results = search_qdrant(question)
    print(f"    Retrieved {len(qdrant_results)} chunks")

    print("  Searching Neo4j...")
    graph_entities = _collect_graph_entities(entities, qdrant_results)
    neo4j_results  = search_neo4j(graph_entities)
    print(f"    Retrieved facts for {len(neo4j_results)} entities")

    # 1 — Vector only
    print(f"\n{thin}")
    print("ANSWER 1 — VECTOR SEARCH ONLY (Qdrant semantic similarity)")
    print(thin)
    print(answer_vector_only(question, entities, qdrant_results))

    # 2 — Graph only
    print(f"\n{thin}")
    print("ANSWER 2 — GRAPH RAG ONLY (Neo4j multi-hop traversal)")
    print(thin)
    print(answer_graph_only(question, neo4j_results))

    # 3 — Hybrid
    print(f"\n{thin}")
    print("ANSWER 3 — HYBRID (Vector + Graph)")
    print(thin)
    print(answer_hybrid(question, qdrant_results, neo4j_results))

## 10 · Run the Demo

Edit the `questions` list below to try your own queries.

**Example questions that showcase the difference between retrieval modes:**
- *"Which scientists worked on nuclear fission and what were their contributions?"* — vector search finds relevant passages; graph search adds explicit collaboration links
- *"How are Marie Curie and Niels Bohr connected through their work?"* — graph multi-hop traversal can trace the chain even if no single passage describes it
- *"Who influenced Alan Turing's thinking in computer science?"* — hybrid mode gives the most complete picture

Each call to `compare_answers()` prints the three answers one after another so you can compare them.

In [20]:
questions = [
    "Who influenced Nikola Tesla, and what fields or concepts did those influences work on?",
    "Which organizations or institutions connect the scientists who worked on radioactivity?",
    "What intermediate concepts or people connect Alan Turing's work to modern artificial intelligence?",
]

try:
    for q in questions:
        compare_answers(q)
finally:
    neo4j_driver.close()


QUESTION: Who influenced Nikola Tesla, and what fields or concepts did those influences work on?
  Extracting entities...
    Found: ['Nikola Tesla', 'fields', 'concepts']
  Searching Qdrant...
    Retrieved 5 chunks
  Searching Neo4j...
    Retrieved facts for 1 entities

--------------------------------------------------------------------------------
ANSWER 1 — VECTOR SEARCH ONLY (Qdrant semantic similarity)
--------------------------------------------------------------------------------
Nikola Tesla was influenced by several individuals and concepts across various fields:

1. **Mark Twain** - A close friend of Tesla, Twain admired Tesla's inventions, notably describing the induction motor as "the most valuable patent since the telephone." Their friendship likely provided Tesla with intellectual and social stimulation.

2. **Swami Vivekananda** - Tesla met the Indian Hindu monk in 1896, which sparked his interest in Eastern science and philosophy. Vivekananda's ideas on Vedantic cos